# 1. Objective
This notebook generates our first synthetic dataset using a Gaussian Copula synthesizer from the SDV library. The generator is fitted strictly on the `adult.data` (training) dataset. We will generate approximately the same number of rows as the original training dataset (32,561 rows) and save it for future utility testing.

# 2. Load Real Training Data

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

columns = [
    "age", "workclass", "fnlwgt", "education", "education-num", 
    "marital-status", "occupation", "relationship", "race", "sex", 
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

# We load ONLY the training dataset. We parse '?' as NaN.
df = pd.read_csv('../data/raw/adult.data', names=columns, skipinitialspace=True, na_values='?')
print(f"Loaded {len(df)} records from real training dataset.")
df.head()

Loaded 32561 records from real training dataset.


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


# 3. Prepare Data for the Generator
We use SDV's `SingleTableMetadata` to automatically infer the types of the columns. SDV is highly flexible and natively handles mixed data types and missing values (NaNs).

In [2]:
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=df)

# 4. Configure Gaussian Copula
We initialize the `GaussianCopulaSynthesizer` using our detected metadata.

In [3]:
from sdv.single_table import GaussianCopulaSynthesizer

synthesizer = GaussianCopulaSynthesizer(
    metadata,
    enforce_min_max_values=True,
    enforce_rounding=True
)

# 5. Fit Generator
We fit the synthesizer purely on the original training dataset.

In [4]:
# This computes the marginal distributions and correlation matrix
synthesizer.fit(df)

# 6. Generate Synthetic Data
We generate the exact same number of rows (32,561) using a fixed seed for reproducibility.

In [5]:
import sdv
print(f"Using SDV version: {sdv.__version__}")

# We use a fixed seed for reproducible generation
import torch
import numpy as np
torch.manual_seed(42)
np.random.seed(42)

synthetic_data = synthesizer.sample(num_rows=len(df))

Using SDV version: 1.38.3


# 7. Basic Sanity Checks
We check dimensions, column names, and a quick summary to ensure the dataset looks structurally identical.

In [6]:
print("--- Real Data Dimensions ---")
print(df.shape)
print("\n--- Synthetic Data Dimensions ---")
print(synthetic_data.shape)

print("\n--- Synthetic Data Missing Values ---")
print(synthetic_data.isnull().sum()[synthetic_data.isnull().sum() > 0])

print("\n--- Synthetic Target Distribution ---")
print(synthetic_data['income'].value_counts(normalize=True))

synthetic_data.head()

--- Real Data Dimensions ---
(32561, 15)

--- Synthetic Data Dimensions ---
(32561, 15)

--- Synthetic Data Missing Values ---
workclass         1770
occupation        1926
native-country     554
dtype: int64

--- Synthetic Target Distribution ---
income
<=50K    0.752864
>50K     0.247136
Name: proportion, dtype: float64


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,55,NaN,151783,9th,12,Divorced,Exec-managerial,Husband,White,Male,13684,0,32,United-States,>50K
1,42,Private,357315,Bachelors,12,Married-civ-spouse,Sales,Unmarried,White,Male,492,0,33,United-States,<=50K
2,24,Private,168438,HS-grad,6,Married-civ-spouse,Craft-repair,Unmarried,Black,Female,76972,18,38,United-States,<=50K
3,24,Private,96982,HS-grad,10,Married-civ-spouse,Adm-clerical,Not-in-family,White,Male,32075,1,32,United-States,<=50K
4,32,Self-emp-not-inc,233587,Assoc-voc,6,Divorced,Tech-support,Husband,White,Male,48677,5,40,France,>50K


# 8. Save Synthetic Dataset
We export the dataset to our `data/synthetic` directory.

In [7]:
import os
os.makedirs('../data/synthetic', exist_ok=True)
synthetic_data.to_csv('../data/synthetic/adult_gaussian_copula.csv', index=False)
print("Saved to: data/synthetic/adult_gaussian_copula.csv")

Saved to: data/synthetic/adult_gaussian_copula.csv


# 9. Initial Observations
- **Generator**: SDV `GaussianCopulaSynthesizer`.
- **Data Handling**: Missing values (`?`) were natively passed to the synthesizer as NaNs. The synthesizer learned the marginal distributions and the covariance.
- **Rows Generated**: 32,561 (matching training data size).
- **Columns**: 15 (matching training data structure).
- **Seed/Version**: Fixed seed used. (Version documented in cell output).